In [2]:
from dandi.dandiapi import DandiAPIClient
import pynwb
import pandas as pd

dandiset_id = "000008"
version = "0.211014.0809"

client = DandiAPIClient()

# returns all NWB files in the dandiset
assets = list(client.get_dandiset(dandiset_id).get_assets())
len(assets)


1328

In [ ]:
"Gene_Expression/Data/expression_matrix.hdf5"

In [5]:
import os, re, h5py
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
from rapidfuzz import process, fuzz

# --- Your paths ---
NWB_PATH = "DandiSetData/000008/sub-mouse-AAYYT/sub-mouse-AAYYT_ses-20180420-sample-4_slice-20180420-slice-4_cell-20180420-sample-4_icephys.nwb"
H5_PATH  = "Gene_Expression/Data/expression_matrix.hdf5"

def normalize_key(s: str | None) -> str | None:
    if s is None: return None
    s = s.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

def nwb_identifiers(nwb_path: str) -> dict:
    """Extract plausible linking strings from a Patch-seq NWB."""
    with NWBHDF5IO(nwb_path, "r", load_namespaces=True) as io:
        nwb = io.read()
        out = {}
        out["identifier"] = getattr(nwb, "identifier", None)
        out["session_id"] = getattr(nwb, "session_id", None)

        subj = getattr(nwb, "subject", None)
        out["subject_id"] = getattr(subj, "subject_id", None) if subj else None

        gen = getattr(nwb, "general", None)
        # common custom keys seen in Patch-seq NWBs
        for k in ["cell_id", "specimen_id", "cell_specimen_id", "slice_id"]:
            out[k] = getattr(gen, k, None) if gen else None

        # filename-derived hints (often encode sample/slice/cell)
        stem = os.path.splitext(os.path.basename(nwb_path))[0]
        out["file_stem"] = stem

        # Build extra candidates by parsing standard tokens
        # e.g., "..._ses-20180420-sample-2_slice-20180420-slice-2_cell-20180420-sample-2_..."
        tokens = re.findall(r"(ses-\d{8}|sample-\d+|slice-\d{8}-slice-\d+|cell-\d{8}-sample-\d+)", stem)
        out["tokens"] = "_".join(tokens) if tokens else None

        # A very common RNA key candidate looks like "20180420_sample_2"
        m_ses  = re.search(r"ses-(\d{8})", stem)
        m_samp = re.search(r"sample-(\d+)", stem)
        if m_ses and m_samp:
            out["date_sample"] = f"{m_ses.group(1)}_sample_{m_samp.group(1)}"

    # add normalized versions
    out_norm = {f"{k}_norm": normalize_key(v) for k, v in out.items() if isinstance(v, str)}
    out.update(out_norm)
    return out


out = nwb_identifiers(NWB_PATH)

/home/peter/.local/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'hdmf-common' version 1.3.0 because version 1.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/peter/.local/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.2.5 because version 2.7.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/peter/.local/lib/python3.10/site-packages/pynwb/icephys.py:226: UserWarning: Stimulus description 'NA' for IZeroClampSeries 'CurrentClampSeries009' is ignored and will be set to 'N/A' as per NWB 2.3.0.
  warnings.warn(


In [6]:
out

{'identifier': '98e1a2a9-d269-4dee-9cb9-b02b219efd21',
 'session_id': '20180420_sample_4',
 'subject_id': 'mouse_AAYYT',
 'cell_id': None,
 'specimen_id': None,
 'cell_specimen_id': None,
 'slice_id': None,
 'file_stem': 'sub-mouse-AAYYT_ses-20180420-sample-4_slice-20180420-slice-4_cell-20180420-sample-4_icephys',
 'tokens': 'ses-20180420_sample-4_slice-20180420-slice-4_cell-20180420-sample-4',
 'date_sample': '20180420_sample_4',
 'identifier_norm': '98e1a2a9_d269_4dee_9cb9_b02b219efd21',
 'session_id_norm': '20180420_sample_4',
 'subject_id_norm': 'mouse_aayyt',
 'file_stem_norm': 'sub_mouse_aayyt_ses_20180420_sample_4_slice_20180420_slice_4_cell_20180420_sample_4_icephys',
 'tokens_norm': 'ses_20180420_sample_4_slice_20180420_slice_4_cell_20180420_sample_4',
 'date_sample_norm': '20180420_sample_4'}

In [4]:
import h5py, re
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix
import anndata as ad

H5_PATH = "Gene_Expression/Data/expression_matrix.hdf5"
RNA_KEY = "20180420_sample_2"   # from NWB session_id/date_sample
NWB_PATH = "DandiSetData/000008/sub-mouse-AAYYT/sub-mouse-AAYYT_ses-20180420-sample-2_slice-20180420-slice-2_cell-20180420-sample-2_icephys.nwb"

# --- 1) Load sample & gene names (no matrix yet) ---
with h5py.File(H5_PATH, "r") as f:
    sample_names = [s.decode() for s in f["sample_names"][:]]
    gene_names   = [g.decode() for g in f["gene_names"][:]]

# Exact membership check (no fuzzy)
assert RNA_KEY in sample_names, f"{RNA_KEY} not found in RNA sample_names."

# --- 2) Load the expression matrix as cells×genes AnnData ---
def load_allen_smartseq_h5(path, group="data/t_exon"):
    with h5py.File(path, "r") as f:
        genes  = [g.decode() for g in f["gene_names"][:]]
        cells  = [s.decode() for s in f["sample_names"][:]]
        grp    = f[group]
        dims   = tuple(grp["dims"][:])
        i      = grp["i"][:]; p = grp["p"][:]; x = grp["x"][:]
        X = csc_matrix((x, i, p), shape=dims)

        # Orient to cells×genes
        ncols = len(p) - 1
        G, C  = len(genes), len(cells)
        if   ncols == G: pass
        elif ncols == C: X = X.T
        else:
            if   dims[0]==C and dims[1]==G: pass
            elif dims[0]==G and dims[1]==C: X = X.T
            else: raise ValueError("Can't infer matrix orientation")
    return ad.AnnData(X=X, obs=pd.DataFrame(index=cells), var=pd.DataFrame(index=genes))

adata = load_allen_smartseq_h5(H5_PATH, group="data/t_exon")

# --- 3) Extract expression vector for this cell ---
row = adata[RNA_KEY, :].X          # sparse row
# Example peek of a few genes
probe = ["Fos","Egr1","Bdnf","Camk2g"]
present = [g for g in probe if g in adata.var_names]
expr = np.asarray(adata[RNA_KEY, present].X.todense()).ravel() if present else np.array([])
print("Matched RNA sample:", RNA_KEY)
print("Genes present:", present)
print("Expression:", expr)

# --- 4) Save a one-line linkage record you can build on later ---
link = pd.DataFrame([{
    "nwb_path": NWB_PATH,
    "nwb_session_id": RNA_KEY,
    "rna_sample_name": RNA_KEY
}])
link.to_csv("patchseq_link_onecell.csv", index=False)
print("Saved linkage -> patchseq_link_onecell.csv")


AssertionError: 20180420_sample_2 not found in RNA sample_names.